# Week 02 · Day 02 — Fine-Tuning with QLoRA

**Topics:** LoRA concepts · QLoRA setup · SFTTrainer · Evaluation · Merge and save

> **GPU required** — this notebook is designed to run on Colab T4 or better. CPU-only will be too slow for actual training, but the setup and evaluation cells still run.

In [ ]:
%pip install -q transformers peft trl bitsandbytes datasets accelerate

## 1 · LoRA Concepts

**LoRA** (Low-Rank Adaptation) freezes the original model weights and adds small trainable matrices at each attention layer. Instead of training all 7B parameters, you train ~0.1%.

Two key hyperparameters:
- **r** (rank): size of the low-rank matrices. Higher r → more parameters → more capacity. Start with 16.
- **lora_alpha**: scaling factor. Convention: set to 2×r (alpha=32 for r=16).

**QLoRA** = LoRA + 4-bit quantization. The base model is loaded in NF4 (4-bit); only the LoRA adapters are stored in float16.

In [ ]:
# Memory breakdown for Llama-3.2-3B with QLoRA
model_params = 3_000_000_000
lora_r = 16
target_modules = 8  # q_proj, k_proj, v_proj, o_proj, gate, up, down (approx)

base_mem_4bit_gb = model_params * 4 / 8 / 1e9  # 4 bits per param

# LoRA params per module: 2 * hidden_size * r (very approximate)
hidden_size = 3072  # for 3B model
lora_params = target_modules * 2 * hidden_size * lora_r
lora_mem_gb = lora_params * 2 / 1e9  # float16
lora_pct = lora_params / model_params * 100

print(f"Base model (4-bit):  {base_mem_4bit_gb:.1f} GB")
print(f"LoRA adapters (fp16): {lora_mem_gb:.3f} GB ({lora_pct:.4f}% of base params)")
print(f"Total VRAM needed:   ~{base_mem_4bit_gb + lora_mem_gb + 2:.1f} GB (incl. optimizer/activations)")

## 2 · QLoRA Setup

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # 1B for demo; use 3B or 8B for real tasks
# Alternative: "TinyLlama/TinyLlama-1.1B-Chat-v1.0" (no HF token needed)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 preserves more information than FP4
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants too
)

print("BitsAndBytesConfig ready")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load model and tokenizer (skip on CPU to save time)
if torch.cuda.is_available():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model)  # enables gradient checkpointing + upcasting

    # LoRA config
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("Skipping model load — no GPU detected")
    print("Run this cell on Colab with GPU runtime")

## 3 · Training Data Format

Instruction-tuned models expect data in the chat template format. Each example is a conversation with `role: user` and `role: assistant` turns.

In [ ]:
from datasets import Dataset

# Raw training examples
raw_examples = [
    {
        "instruction": "Classify the sentiment of this review as Positive or Negative.",
        "input": "The product arrived broken and customer service was unhelpful.",
        "output": "Negative",
    },
    {
        "instruction": "Classify the sentiment of this review as Positive or Negative.",
        "input": "Absolutely love this! Best purchase I've made all year.",
        "output": "Positive",
    },
    {
        "instruction": "Classify the sentiment of this review as Positive or Negative.",
        "input": "Decent quality but the price is too high for what you get.",
        "output": "Negative",
    },
]

def format_example(example: dict) -> dict:
    text = (
        f"<|system|>\n{example['instruction']}\n"
        f"<|user|>\n{example['input']}\n"
        f"<|assistant|>\n{example['output']}"
    )
    return {"text": text}

formatted = [format_example(e) for e in raw_examples]
dataset = Dataset.from_list(formatted)

print("Example formatted training record:")
print(dataset[0]["text"])

## 4 · SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

if torch.cuda.is_available():
    training_args = SFTConfig(
        output_dir="./qlora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 4 * 4 = 16
        gradient_checkpointing=True,      # trade compute for VRAM
        learning_rate=2e-4,
        bf16=True,                        # use bfloat16 on Ampere+ GPUs
        logging_steps=10,
        save_steps=100,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        max_seq_length=512,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )

    # trainer.train()  # uncomment to actually train
    print("SFTTrainer configured — call trainer.train() to start")
else:
    print("SFTTrainer requires GPU. Pattern shown:")
    print("""
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(output_dir='./out', num_train_epochs=3, ...),
        train_dataset=dataset,
        tokenizer=tokenizer,
    )
    trainer.train()
    """)

## 5 · Inference with Fine-Tuned Adapter and Merge

In [ ]:
# After training: load adapter and run inference
# from peft import PeftModel
# base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, ...)
# model = PeftModel.from_pretrained(base_model, "./qlora-output")

# Merge adapter into base model for faster inference
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./merged-model")
# tokenizer.save_pretrained("./merged-model")

print("Merge pattern (run after training):")
print("""
  from peft import PeftModel

  base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
  model = PeftModel.from_pretrained(base, './qlora-output')
  merged = model.merge_and_unload()          # fuse LoRA weights into base
  merged.save_pretrained('./merged-model')   # standard HuggingFace format
  tokenizer.save_pretrained('./merged-model')
""")

In [ ]:
# Evaluate: compare fine-tuned vs base model on held-out test set
def evaluate_classification(model, tokenizer, test_examples: list[dict]) -> float:
    correct = 0
    for ex in test_examples:
        prompt = f"<|system|>\n{ex['instruction']}\n<|user|>\n{ex['input']}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=5, temperature=0)
        pred = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if pred.lower().startswith(ex["output"].lower()):
            correct += 1
    return correct / len(test_examples)

print("evaluate_classification() defined")
print("Usage: accuracy = evaluate_classification(model, tokenizer, test_examples)")
print("Compare: base model accuracy vs fine-tuned accuracy to measure improvement")

## Summary

- **LoRA**: adds trainable low-rank matrices; freezes base weights. Only ~0.1% of params trained.
- **QLoRA**: loads base in 4-bit NF4; LoRA adapters in fp16. Trains 7B models on 8–10GB VRAM.
- **r=16, alpha=32**: the standard starting point. Increase r for more complex tasks.
- **SFTTrainer**: handles chat template formatting, gradient checkpointing, mixed precision automatically.
- **merge_and_unload()**: fuse LoRA weights into base for deployment — removes adapter dependency.

**Next:** [Function Calling](week02-day02-function-calling.ipynb)